# Greedy String Art

Preprocess a target image, precalc the line matrix, then greedily select lines to approximate it.

In [ ]:
import matrix
import greedy
from preprocess import circle_crop
from PIL import Image
from importlib import reload
import matplotlib.pyplot as plt
import numpy as np

reload(matrix)
reload(greedy)

## Settings

Edit image path, resolution, peg count, and line budget here.

In [ ]:
H = 256
P = 120
LINE_WIDTH = 0.2
MAX_LINES = 2000
CIRCLE_PADDING = 32
IMAGE_PATH = 'razer/tiger2.jpg'

## Preprocess target

Crop to a circle on a white background and convert to grayscale.

In [ ]:
target = Image.open(IMAGE_PATH)
target = circle_crop(target, H, CIRCLE_PADDING).convert('L')
print(np.array(target).shape)

plt.imshow(target, cmap='gray')
plt.axis('off')
plt.show()

## Precalc line matrix

Build the antialiased darkness matrix `M` (one column per peg-to-peg line).

In [ ]:
M = matrix.precalc(H, P, line_width=LINE_WIDTH, sparse=False)
print('M shape:', M.shape)

## Greedy solve

Select lines one at a time (binary 0/1) to minimize MSE against the target.

In [ ]:
x, selected = greedy.greedy_solve(
    M,
    np.array(target),
    H,
    max_lines=MAX_LINES,
    line_width=LINE_WIDTH,
    verbose=True,
)

print(f'Selected {len(selected)} lines')

## Render result

Compare the target with the greedy reconstruction.

In [ ]:
result = matrix.render_image(M, x, H, threshold=0.5, line_width=LINE_WIDTH)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(target, cmap='gray')
axes[0].set_title('Target')
axes[0].axis('off')

axes[1].imshow(result, cmap='gray')
axes[1].set_title(f'Greedy ({len(selected)} lines)')
axes[1].axis('off')

plt.tight_layout()
plt.show()